In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("..")

import numpy as np
from numpy.linalg import norm
from datetime import datetime
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from tqdm import tqdm
from dataclasses import dataclass
from lunanav.constants import *  # noqa: F403
from lunanav.sim.simulator import SimParams, RigidBody, run_sim, SimResults, reverse_sim_results
from lunanav.sim.sensors import (
    SensorEnvironment, Sensor, SensorSuite,
    accelerometer_sensor, gyroscope_sensor,
    laser_altimeter_sensor, laser_velocity_sensor,
    star_tracker_sensor, doppler_sensor, sat_range_tracker_sensor
)
from lunanav.estimation.ekf import ekf_predict, ekf_update, Qd_from_accel_white, update_sensor, update_sensor_individual_NaN_check
from lunanav.sim.quaternion import unitize_state, angle_axis_to_q, quat_apply, conj
from lunanav.plotting import plot_control_effort, plot_state_vector_combined, plot_state_vector_combined_log, plot_state_vector
from lunanav.visualization import (
    visualize_trajectory, plot_measurements, plot_attitude_relative_vertical, analyze_ekf_error,
    plot_filter_confidence, obsv_verbose, plot_satellites_3d_plotly, plot_accelerometer, plot_gyroscope)
from lunanav.loaders import load_trajectory, load_ekf_result, save_ekf_result, save_trajectory, save_sim_result, load_sim_result, EKFResult, Trajectory, SimResult
from lunanav.sim.sensors import get_los_vectors
from lunanav.sim.generate import SatPosVel, make_sat_arrs, generate_env, generate_measurements


In [ ]:
lander = RigidBody(
    mass_kg=50,
    I=np.diag([8,8,5])
)
dt = 0.1
t_max = 200.0

In [ ]:
sensor_suite = SensorSuite(sensors={
    "accelerometer": accelerometer_sensor(sigma_accel),
    "gyroscope": gyroscope_sensor(sigma_gyro),
    "laser_altimeter": laser_altimeter_sensor(sigma_los),
    "laser_velocity": laser_velocity_sensor(sigma_los_vel),
    "star_tracker": star_tracker_sensor(sigma_star),
    "doppler": doppler_sensor(3, sigma_doppler),
    "range_tracker": sat_range_tracker_sensor(3, sigma_sat_range_tracker),
})

# EKF with Sequential Sensor Updates

In [ ]:
# sim_save_path = "traj_A_skew.json" # skew
sim_save_path = "traj_B_liftoff.json" # skew
# sim_save_path = "traj_C_piecewise.json" # piecewise
sim_result = load_sim_result(f"data/simresults/{sim_save_path}")

# s_true   = sim_result.s_true
# t        = sim_result.t_arr
dt       = sim_result.dt
n        = sim_result.nsteps
# mass_kg  = sim_result.mass_kg
# I        = sim_result.I.reshape((3, 3))a

results = SimResults(n)


results.t = sim_result.t
results.states = sim_result.s_arr
results.force_N = sim_result.force
results.torque_Nm = sim_result.torque

lander = RigidBody(
    mass_kg=sim_result.mass_kg,
    I=sim_result.I
)
sim = SimParams(results.states[0], lander, dt, t_max)
doppler_sats: list[SatPosVel] = [
    make_sat_arrs(results.t, altitude=100e3, raan=0, aop=90, inc=90), # overhead going -x
    make_sat_arrs(results.t, altitude=100e3, raan=90, aop=94, inc=94),
    make_sat_arrs(results.t, altitude=100e3, raan=160, aop=70, inc=86),
]
env_arr = generate_env(results, sim, doppler_sats)


measurements_clean = {
    k: np.array(v["truth"]) for k, v in sim_result.measurements.items()
}
measurements_noisy = {
    k: np.array(v["noisy"]) for k, v in sim_result.measurements.items()
}
print(f"Loaded sim result: {n} steps, dt={dt}s")

print(f"{results.states[0] - MOON_13_VEC(1)=}")
print(f"{np.max(np.linalg.norm(results.force_N, axis=1))=} N")
print(f"{np.max(np.linalg.norm(results.torque_Nm, axis=1))=} Nm")
print(f"{sim_save_path=}")
# visualize_trajectory(results.states, results.t, dt, offset = MOON_3_VEC(1)[0], title="Traj", show_lander=False, downsample_rate=20, moon_resolution = 35).show()

In [ ]:
fig = plot_measurements(measurements_clean, measurements_noisy, results, sensor_suite)
plt.show()

In [ ]:
doppler_sats: list[SatPosVel] = [
    make_sat_arrs(results.t, altitude=100e3, raan=0, aop=90, inc=90), # overhead going -x
    make_sat_arrs(results.t, altitude=100e3, raan=90, aop=94, inc=94),
    make_sat_arrs(results.t, altitude=100e3, raan=160, aop=70, inc=86),
]
env_arr = generate_env(results, sim, doppler_sats)
measurements_clean, measurements_noisy = generate_measurements(results.states, env_arr, sensor_suite)

In [ ]:
# EKF setup



mu_arr = np.zeros((n, 13))
Sigma_arr = np.zeros((n, 13, 13))


In [ ]:
seed = 178
rng = np.random.default_rng(seed)

state0 = results.states[0]


# Process noise
Q_ekf = np.zeros((13, 13))
Q_ekf[0:6, 0:6] = Qd_from_accel_white(dt, sigma_accel)
Q_ekf[6:10, 6:10] = np.eye(4) * 1e-6
Q_ekf[10:13, 10:13] = np.eye(3) * sigma_gyro**2 * dt
offset = rng.multivariate_normal(np.zeros(13), Q_ekf * 100000)
print(f"Offset: {offset}")
# Initial state estimate (slightly off from truth)
mu_arr[0] = state0
# mu_arr[0] = state0 + np.array([1e3, -1e3, 1e3, 0, 0, 0, *angle_axis_to_q(0, [1,0,0], True), 0, 0, 0])
mu_arr[0] = state0 + np.array([1e3, -1e3, 1e3, 120, 20, 46, *angle_axis_to_q(30, [0,1,0], True), 10 * DEG_TO_RAD, -20 * DEG_TO_RAD, 30 * DEG_TO_RAD])
# mu_arr[0] = state0 + offset
mu_arr[0] = unitize_state(mu_arr[0])
Sigma_arr[0] = np.eye(13)  # small initial uncertainty





print("EKF initialized")
print(f"Initial position error: {norm(mu_arr[0, 0:3] - state0[0:3]):.2f} m")

print(angle_axis_to_q(20, [0,1,0], True))

In [ ]:
# Plot true trajectory state vector

# Extract state components
r_true = results.states[:, 0:3]      # position
v_true = results.states[:, 3:6]      # velocity
q_true = results.states[:, 6:10]     # quaternion
w_true = results.states[:, 10:13]    # angular velocity

print(f"State vector ranges:")
print(f"  Position:  X=[{r_true[:, 0].min():.0f}, {r_true[:, 0].max():.0f}], Y=[{r_true[:, 1].min():.0f}, {r_true[:, 1].max():.0f}], Z=[{r_true[:, 2].min():.0f}, {r_true[:, 2].max():.0f}]")
print(f"  Velocity:  Vx=[{v_true[:, 0].min():.1f}, {v_true[:, 0].max():.1f}], Vy=[{v_true[:, 1].min():.1f}, {v_true[:, 1].max():.1f}], Vz=[{v_true[:, 2].min():.1f}, {v_true[:, 2].max():.1f}]")
print(f"  Quaternion norm (should be 1): min={np.linalg.norm(q_true, axis=1).min():.6f}, max={np.linalg.norm(q_true, axis=1).max():.6f}")
print(f"  Angular velocity: ωx=[{w_true[:, 0].min():.4f}, {w_true[:, 0].max():.4f}], ωy=[{w_true[:, 1].min():.4f}, {w_true[:, 1].max():.4f}], ωz=[{w_true[:, 2].min():.4f}, {w_true[:, 2].max():.4f}]")


In [ ]:
# DIAGNOSTICS: Check trajectory and measurement dimensions
print("=== TRAJECTORY DIAGNOSTICS ===")
# print(f"liftoff_results.nsteps: {liftoff_results.nsteps}")
# print(f"landing_results.nsteps: {landing_results.nsteps}")
print(f"n (current): {n}")
print(f"results.t length: {len(results.t)}")
print(f"results.states shape: {results.states.shape}")
print(f"results.force_N shape: {results.force_N.shape}")

print("\nMeasurement arrays shape:")
for sensor_name, meas_array in measurements_clean.items():
    print(f"  {sensor_name}: {meas_array.shape}")

print("\nEKF arrays shape:")
print(f"  mu_arr: {mu_arr.shape}")
print(f"  Sigma_arr: {Sigma_arr.shape}")

print("\nInitial state check:")
print(f"  state0 shape: {state0.shape}")
print(f"  state0: {state0}")
print(f"  mu_arr[0]: {mu_arr[0]}")

In [ ]:


sensor_frequencies_cases = [
    {
        "name": "All 10Hz",
        "frequencies": {"laser_altimeter": 10, "laser_velocity": 10, 
                       "star_tracker": 10, "doppler": 10, "range_tracker": 10}
    },
    {
        "name": "LOS 2 Hz",
        "frequencies": {"laser_altimeter": 50, "laser_velocity": 50, 
                       "star_tracker": 10, "doppler": 10, "range_tracker": 10}
    },
    {
        "name": "Low attitude (1 Hz star tracker)",
        "frequencies": {"laser_altimeter": 10, "laser_velocity": 10, 
                       "star_tracker": 100, "doppler": 10, "range_tracker": 10}
    },
    {
        "name": "1 Hz Constellation, no laser",
        "frequencies": {"laser_altimeter": None, "laser_velocity": None, 
                       "star_tracker": 10, "doppler": 100, "range_tracker": 100}
    },
    {
        "name": "Very low attitude (0.5 Hz star tracker)",
        "frequencies": {"laser_altimeter": 10, "laser_velocity": 10, 
                       "star_tracker": 200, "doppler": 10, "range_tracker": 10}
    },
    {
        "name": "Laser 50 Hz, 0.5 Hz other sensors",
        "frequencies": {"laser_altimeter": 2, "laser_velocity": 2, 
                       "star_tracker": 200, "doppler": 200, "range_tracker": 200}
    }
]

In [ ]:
valid = True

# Run EKF with new sensor architecture using NOISY measurements
# Sequential updates: predict, then update with available sensors

freq_results = []

for case in sensor_frequencies_cases:


    for i in tqdm(range(n - 1)):

        accel_meas = measurements_noisy["accelerometer"][i]  # noisy
        gyro_meas = measurements_noisy["gyroscope"][i]       # noisy
        # print(f"{accel_meas=}")
        # print(f"{gyro_meas=}")
        # print()
        # print(f"Before: {mu_arr[i]=}")
        mu_pred, Sigma_pred = ekf_predict(mu_arr[i], Sigma_arr[i], accel_meas, gyro_meas, Q_ekf, sim)
        mu_pred = unitize_state(mu_pred)
        # print(f"AFter: {mu_pred=}")

        env = env_arr[i]

        if jnp.any(jnp.isnan(mu_pred)):
            print(f"NaN after predict at i={i}")
            valid = False


        for sensor, freq in case["frequencies"].items():
            if sensor in ["laser_altimeter", "laser_velocity"]:
                mu_pred, Sigma_pred = update_sensor_individual_NaN_check(sensor, freq, mu_pred, Sigma_pred, env, sensor_suite, measurements_noisy, i)
            else:
                mu_pred, Sigma_pred = update_sensor(sensor, freq, mu_pred, Sigma_pred, env, sensor_suite, measurements_noisy, i)

            if jnp.any(jnp.isnan(mu_pred)):
                print(f"NaN after update for {sensor} at i={i}")
                valid = False
        mu_arr[i + 1] = mu_pred
        Sigma_arr[i + 1] = Sigma_pred

        if not valid:
            break


    pos_error = np.linalg.norm(mu_arr[:, 0:3] - results.states[:, 0:3], axis=1)
    vel_error = np.linalg.norm(mu_arr[:, 3:6] - results.states[:, 3:6], axis=1)
    att_error = np.linalg.norm(mu_arr[:, 6:10] - results.states[:, 6:10], axis=1)
    
    freq_results.append({
        "name": case['name'],
        "mu_arr": mu_arr.copy(),
        "Sigma_arr": Sigma_arr.copy(),
        "pos_error": pos_error.copy(),
        "vel_error": vel_error.copy(),
        "att_error": att_error.copy(),
    })

print("EKF completed")

In [ ]:
# # Standard per-case analysis
# for r in freq_results:
#     fig, stats = analyze_ekf_error(
#         results, r["mu_arr"], results.t,
#         case_name=r["name"],
#         save_path=f"data/error_analysis/case4_{r["name"]}.png"
#     )
#     plt.show()


In [ ]:
plt.semilogy(norm(freq_results[0]["mu_arr"][:, 0:3] - results.states[:, 0:3], axis=1))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(22, 5))
fig.suptitle('Liftoff EKF — Sensor Update Comparison', fontsize=14, fontweight='bold')
colors = plt.cm.tab10(np.linspace(0, 1, len(freq_results)))

for i, (r,case) in enumerate(zip(freq_results, sensor_frequencies_cases)):
    pos_err = norm(r["mu_arr"][:, 0:3] - results.states[:, 0:3], axis=1)
    vel_err = norm(r["mu_arr"][:, 3:6] - results.states[:, 3:6], axis=1)
    att_err = norm(r["mu_arr"][:, 6:10] - results.states[:, 6:10], axis=1)
    
    kw = dict(label=case['name'], linewidth=2)
    axs[0].semilogy(results.t, pos_err, **kw)
    axs[1].semilogy(results.t, vel_err, **kw)
    axs[2].semilogy(results.t, att_err, **kw)

for ax, (ylabel, title) in zip(axs, [
    ('Position Error (m)',  'Position Error'),
    ('Velocity Error (m/s)', 'Velocity Error'),
    ('Attitude Error',       'Attitude Error'),
]):
    ax.set_xlabel('Time (s)'); ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.3, which='both')

handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center right', bbox_to_anchor=(1.0, 0.5), fontsize=9, framealpha=0.9)
plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.savefig("data/error_analysis/case4_imperfect_worst_case.png", dpi=150, bbox_inches='tight')
plt.show()

